# 📹 Notebook 01 — Wykrywanie Punktów Sylwetki
## Pose Landmark Detection from Video

This notebook corresponds to Section 5 of the research report.  
It uses **MediaPipe Pose Landmarker** + **OpenCV** to extract 33 skeletal keypoints from each frame of a video file.

**Output:** A CSV file with columns `frame, timestamp_ms, landmark_id, landmark_name, x, y, z, visibility`

---
**Pipeline position:**
```
Video → [THIS NOTEBOOK] → Landmark CSV → Feature Engineering → Model
```

In [ ]:
import sys
sys.path.insert(0, '..')  # allow importing from src/

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

print('OpenCV version:', cv2.__version__)
print('NumPy version:', np.__version__)
print('Pandas version:', pd.__version__)

## 1. Configuration

Set the paths to your video file and the MediaPipe model below.  
If you don't have a video file, use **Demo Mode** in Section 3.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
MODEL_PATH    = '../pose_landmarker_full.task'   # download with scripts/download_model.py
VIDEO_PATH    = '../data/raw/blender/fall_01.mp4'  # your video file
OUTPUT_CSV    = '../data/landmarks/blender/fall_01.csv'
VIDEO_LABEL   = 'fall_down'     # class label: 'fall_down', 'walking', etc.
VIDEO_SOURCE  = 'blender'       # 'blender', 'kaggle', or 'synthetic'

# Set to True to visualise annotations (slower)
DRAW_LANDMARKS = True

# ── Check paths ────────────────────────────────────────────────────────────
model_exists = Path(MODEL_PATH).exists()
video_exists = Path(VIDEO_PATH).exists()

print(f'Model:  {MODEL_PATH} → {"✓ Found" if model_exists else "✗ Not found"}')
print(f'Video:  {VIDEO_PATH} → {"✓ Found" if video_exists else "✗ Not found"}')

if not model_exists:
    print('\n⚠ Run: python scripts/download_model.py')
if not video_exists:
    print('\n⚠ Place a video file at the path above, or jump to Section 3 (Demo Mode)')

## 2. Extract Landmarks from Video

The `PoseExtractor` class processes the video frame-by-frame using MediaPipe's `VIDEO` running mode, which applies temporal smoothing across frames (better than per-image mode for video).

In [ ]:
from src.extraction.pose_extractor import PoseExtractor, LANDMARK_NAMES

# This cell requires both MODEL_PATH and VIDEO_PATH to exist
if model_exists and video_exists:
    extractor = PoseExtractor(
        model_path=MODEL_PATH,
        draw_landmarks=DRAW_LANDMARKS,
    )

    df = extractor.extract(
        video_path=VIDEO_PATH,
        label=VIDEO_LABEL,
        source=VIDEO_SOURCE,
    )

    print(f'Extracted {len(df):,} rows')
    print(f'Frames processed: {df["frame"].nunique()}')
    print(f'Landmarks per frame: {df["landmark_id"].nunique()}')
    df.head()
else:
    print('Skipping extraction — see Demo Mode below (Section 3)')

In [ ]:
# Save to CSV
if 'df' in dir() and not df.empty:
    out = Path(OUTPUT_CSV)
    out.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out, index=False)
    print(f'✓ Saved to {out}')

## 3. Demo Mode — Synthetic Landmarks (no video needed)

If you don't have a video file, this section generates a realistic synthetic sequence.

In [ ]:
from src.data.generator import SyntheticGenerator

gen = SyntheticGenerator(seed=42)

# Generate one normal and one anomaly sequence for demonstration
df_demo = gen.generate(n_normal=2, n_anomaly=2, seq_duration_s=3.0)

print(f'Generated {len(df_demo):,} rows')
print(f'Unique sequences: {df_demo["sequence_id"].nunique()}')
print(f'Label distribution:\n{df_demo.groupby(["sequence_id", "motion_type", "label"]).size().reset_index(name="rows")}')
df_demo.head(10)

## 4. Visualise Landmark Trajectories

Plot the y-coordinate of the nose and hip midpoint over time.  
In fall sequences, these converge rapidly — the body drops toward the ground.

In [ ]:
plt.style.use('dark_background')
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Landmark Y-Trajectories per Sequence', fontsize=14, color='white', y=1.01)

seq_ids = df_demo['sequence_id'].unique()
colors = {'nose': '#00BCD4', 'left_hip': '#FF9800', 'right_hip': '#FF5722'}

for ax, seq_id in zip(axes.flat, seq_ids):
    seq_df = df_demo[df_demo['sequence_id'] == seq_id]
    motion  = seq_df['motion_type'].iloc[0]
    label   = 'ANOMALY' if seq_df['label'].iloc[0] == 1 else 'NORMAL'
    color   = '#F44336' if label == 'ANOMALY' else '#4CAF50'

    for lm_id, lm_name, clr in [(0, 'nose', '#00BCD4'), (23, 'left_hip', '#FF9800')]:
        lm_df = seq_df[seq_df['landmark_id'] == lm_id].sort_values('frame')
        ax.plot(lm_df['frame'], lm_df['y'], label=lm_name, color=clr, lw=1.5)

    ax.set_title(f'Seq {seq_id} — {motion} [{label}]', color=color, fontsize=10)
    ax.set_xlabel('Frame', color='#aaa')
    ax.set_ylabel('Y (normalized)', color='#aaa')
    ax.set_ylim(0, 1.1)
    ax.invert_yaxis()  # y=0 is top of image
    ax.legend(fontsize=8)
    ax.set_facecolor('#1a1a2e')
    ax.tick_params(colors='#aaa')

plt.tight_layout()
plt.savefig('../reports/landmark_trajectories.png', dpi=150, bbox_inches='tight',
            facecolor='#0d0d1a')
plt.show()
print('Figure saved to reports/landmark_trajectories.png')

## 5. Code from the Research Report (Section 5)

This is the exact code block from the report — the complete extraction loop.

In [ ]:
# ── Fragment kodu z raportu badawczego (Sekcja 5) ──────────────────────────
# This reproduces the code shown in the research brief.

if model_exists and video_exists:
    import mediapipe as mp
    from mediapipe.tasks import python as mp_python
    from mediapipe.tasks.python import vision as mp_vision
    from mediapipe.tasks.python.vision import RunningMode as VisionRunningMode
    from mediapipe.tasks.python.vision import PoseLandmarkerOptions
    from mediapipe.tasks.python.core.base_options import BaseOptions

    model_path = MODEL_PATH
    all_landmarks = []

    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=VisionRunningMode.VIDEO,
        num_poses=1
    )

    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frame_idx = 0

    with mp_vision.PoseLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            timestamp_ms = int((frame_idx / fps) * 1000)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

            result = landmarker.detect_for_video(mp_image, timestamp_ms)

            if result.pose_landmarks:
                landmarks = result.pose_landmarks[0]
                for i, lm in enumerate(landmarks):
                    all_landmarks.append({
                        'frame': frame_idx,
                        'landmark_id': i,
                        'x': lm.x,
                        'y': lm.y,
                        'z': lm.z,
                        'visibility': lm.visibility
                    })

            frame_idx += 1

    cap.release()
    df_raw = pd.DataFrame(all_landmarks)
    print(f'Extracted {len(df_raw):,} landmark observations from {frame_idx} frames.')
    df_raw.head()
else:
    print('Skipping — no video/model found. See Section 3 (Demo Mode).')